In [1]:
from __future__ import annotations

import csv
import os
from datetime import UTC, datetime
from getpass import getpass
from io import StringIO
from typing import Any

import httpx
import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
stocks = [
    {
        "symbol": "AAPL",
        "company_name": "Apple Inc.",
        "sector": "Technology",
    },
    {
        "symbol": "MSFT",
        "company_name": "Microsoft Corporation",
        "sector": "Technology",
    },
    {
        "symbol": "NVDA",
        "company_name": "NVIDIA Corporation",
        "sector": "Semiconductors",
    },
    {
        "symbol": "TSLA",
        "company_name": "Tesla Inc.",
        "sector": "Automotive",
    },
    {
        "symbol": "AMZN",
        "company_name": "Amazon.com Inc.",
        "sector": "Consumer / Cloud",
    },
]

In [5]:
def fetch_yahoo_chart(
    symbol: str,
    *,
    range_value: str = "1mo",
    interval: str = "1d",
) -> dict[str, Any]:
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"

    params = {
        "range": range_value,
        "interval": interval,
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
    }

    with httpx.Client(
        timeout=30.0,
        follow_redirects=True,
        headers=headers,
    ) as http_client:
        response = http_client.get(url, params=params)
        response.raise_for_status()
        data = response.json()

    chart = data.get("chart", {})
    error = chart.get("error")

    if error is not None:
        raise RuntimeError(f"Yahoo Finance returned error for {symbol}: {error}")

    results = chart.get("result", [])

    if not results:
        raise RuntimeError(f"Yahoo Finance returned no chart result for {symbol}")

    return results[0]

In [6]:
aapl_chart = fetch_yahoo_chart("AAPL")

print(aapl_chart.keys())

dict_keys(['meta', 'timestamp', 'indicators'])


In [7]:
print(aapl_chart["meta"])

{'currency': 'USD', 'symbol': 'AAPL', 'exchangeName': 'NMS', 'fullExchangeName': 'NasdaqGS', 'instrumentType': 'EQUITY', 'firstTradeDate': 345479400, 'regularMarketTime': 1780430401, 'hasPrePostMarketData': True, 'gmtoffset': -14400, 'timezone': 'EDT', 'exchangeTimezoneName': 'America/New_York', 'regularMarketPrice': 315.2, 'fiftyTwoWeekHigh': 315.45, 'fiftyTwoWeekLow': 195.07, 'regularMarketDayHigh': 315.45, 'regularMarketDayLow': 306.72, 'regularMarketVolume': 44374534, 'longName': 'Apple Inc.', 'shortName': 'Apple Inc.', 'chartPreviousClose': 280.14, 'priceHint': 2, 'currentTradingPeriod': {'pre': {'timezone': 'EDT', 'end': 1780493400, 'start': 1780473600, 'gmtoffset': -14400}, 'regular': {'timezone': 'EDT', 'end': 1780516800, 'start': 1780493400, 'gmtoffset': -14400}, 'post': {'timezone': 'EDT', 'end': 1780531200, 'start': 1780516800, 'gmtoffset': -14400}}, 'dataGranularity': '1d', 'range': '1mo', 'validRanges': ['1d', '5d', '1mo', '3mo', '6mo', '1y', '2y', '5y', '10y', 'ytd', 'max

In [8]:
def parse_yahoo_chart(chart: dict[str, Any]) -> list[dict[str, Any]]:
    timestamps = chart.get("timestamp", [])

    quote = chart["indicators"]["quote"][0]

    opens = quote.get("open", [])
    highs = quote.get("high", [])
    lows = quote.get("low", [])
    closes = quote.get("close", [])
    volumes = quote.get("volume", [])

    rows = []

    for index, timestamp in enumerate(timestamps):
        open_price = opens[index]
        high_price = highs[index]
        low_price = lows[index]
        close_price = closes[index]
        volume = volumes[index]

        if open_price is None:
            continue

        if high_price is None:
            continue

        if low_price is None:
            continue

        if close_price is None:
            continue

        if volume is None:
            volume = 0

        rows.append(
            {
                "date": datetime.fromtimestamp(timestamp, tz=UTC),
                "open": float(open_price),
                "high": float(high_price),
                "low": float(low_price),
                "close": float(close_price),
                "volume": float(volume),
            }
        )

    return sorted(rows, key=lambda item: item["date"])

In [9]:
aapl_rows = parse_yahoo_chart(aapl_chart)

print("Rows:", len(aapl_rows))
print(aapl_rows[-5:])

Rows: 21
[{'date': datetime.datetime(2026, 5, 27, 13, 30, tzinfo=datetime.timezone.utc), 'open': 308.3299865722656, 'high': 313.260009765625, 'low': 308.29998779296875, 'close': 310.8500061035156, 'volume': 50430900.0}, {'date': datetime.datetime(2026, 5, 28, 13, 30, tzinfo=datetime.timezone.utc), 'open': 310.67999267578125, 'high': 312.79998779296875, 'low': 309.57000732421875, 'close': 312.510009765625, 'volume': 48220400.0}, {'date': datetime.datetime(2026, 5, 29, 13, 30, tzinfo=datetime.timezone.utc), 'open': 311.7799987792969, 'high': 315.0, 'low': 309.5299987792969, 'close': 312.05999755859375, 'volume': 70026800.0}, {'date': datetime.datetime(2026, 6, 1, 13, 30, tzinfo=datetime.timezone.utc), 'open': 309.6300048828125, 'high': 310.94000244140625, 'low': 305.0199890136719, 'close': 306.30999755859375, 'volume': 48849900.0}, {'date': datetime.datetime(2026, 6, 2, 13, 30, tzinfo=datetime.timezone.utc), 'open': 307.4599914550781, 'high': 315.45001220703125, 'low': 306.69000244140625

In [10]:
def get_last_trading_days(
    rows: list[dict[str, Any]],
    days: int = 10,
) -> list[dict[str, Any]]:
    sorted_rows = sorted(rows, key=lambda item: item["date"])
    return sorted_rows[-days:]

In [11]:
def percentage_change(
    previous_close: float,
    current_close: float,
) -> float:
    if previous_close == 0:
        return 0.0

    return ((current_close - previous_close) / previous_close) * 100

In [12]:
def normalize_stock_records(
    *,
    stock_config: dict[str, str],
    rows: list[dict[str, Any]],
    days: int = 10,
) -> list[dict[str, Any]]:
    selected_rows = get_last_trading_days(rows, days=days)

    records = []

    for index, row in enumerate(selected_rows):
        previous_close = selected_rows[index - 1]["close"] if index > 0 else row["open"]

        daily_change_pct = percentage_change(
            previous_close=previous_close,
            current_close=row["close"],
        )

        intraday_range_pct = percentage_change(
            previous_close=row["low"],
            current_close=row["high"],
        )

        if daily_change_pct >= 2:
            trend_label = "strong gain"
        elif daily_change_pct >= 0.5:
            trend_label = "moderate gain"
        elif daily_change_pct <= -2:
            trend_label = "strong loss"
        elif daily_change_pct <= -0.5:
            trend_label = "moderate loss"
        else:
            trend_label = "flat or small move"

        volatility_label = (
            "high intraday range"
            if intraday_range_pct >= 3
            else "normal intraday range"
        )

        summary = (
            f"{stock_config['company_name']} ({stock_config['symbol']}) traded on "
            f"{row['date'].date()} with open {row['open']:.2f}, high {row['high']:.2f}, "
            f"low {row['low']:.2f}, close {row['close']:.2f}, volume {row['volume']:.0f}. "
            f"The daily change was {daily_change_pct:.2f}% and the intraday range was "
            f"{intraday_range_pct:.2f}%. This session is classified as {trend_label} "
            f"with {volatility_label}."
        )

        records.append(
            {
                "symbol": stock_config["symbol"],
                "company_name": stock_config["company_name"],
                "sector": stock_config["sector"],
                "trade_date": row["date"],
                "open_price": row["open"],
                "high_price": row["high"],
                "low_price": row["low"],
                "close_price": row["close"],
                "volume": row["volume"],
                "daily_change_pct": round(daily_change_pct, 4),
                "intraday_range_pct": round(intraday_range_pct, 4),
                "trend_label": trend_label,
                "volatility_label": volatility_label,
                "summary": summary,
            }
        )

    return records

In [13]:
stock_records = []

for stock_config in stocks:
    print("Fetching:", stock_config["symbol"])

    chart = fetch_yahoo_chart(
        stock_config["symbol"],
        range_value="1mo",
        interval="1d",
    )

    rows = parse_yahoo_chart(chart)

    records = normalize_stock_records(
        stock_config=stock_config,
        rows=rows,
        days=10,
    )

    stock_records.extend(records)

print("Total records:", len(stock_records))

Fetching: AAPL
Fetching: MSFT
Fetching: NVDA
Fetching: TSLA
Fetching: AMZN
Total records: 50


In [14]:
for record in stock_records[:5]:
    print(record)
    print("-" * 80)

{'symbol': 'AAPL', 'company_name': 'Apple Inc.', 'sector': 'Technology', 'trade_date': datetime.datetime(2026, 5, 19, 13, 30, tzinfo=datetime.timezone.utc), 'open_price': 296.9700012207031, 'high_price': 300.510009765625, 'low_price': 296.3500061035156, 'close_price': 298.9700012207031, 'volume': 42243600.0, 'daily_change_pct': 0.6735, 'intraday_range_pct': 1.4037, 'trend_label': 'moderate gain', 'volatility_label': 'normal intraday range', 'summary': 'Apple Inc. (AAPL) traded on 2026-05-19 with open 296.97, high 300.51, low 296.35, close 298.97, volume 42243600. The daily change was 0.67% and the intraday range was 1.40%. This session is classified as moderate gain with normal intraday range.'}
--------------------------------------------------------------------------------
{'symbol': 'AAPL', 'company_name': 'Apple Inc.', 'sector': 'Technology', 'trade_date': datetime.datetime(2026, 5, 20, 13, 30, tzinfo=datetime.timezone.utc), 'open_price': 298.17999267578125, 'high_price': 302.79998

In [15]:
record = stock_records[0]

print("trade_date:", record["trade_date"])
print("trade_date type:", type(record["trade_date"]))
print("trade_date tzinfo:", record["trade_date"].tzinfo)

print("volume:", record["volume"])
print("volume type:", type(record["volume"]))

trade_date: 2026-05-19 13:30:00+00:00
trade_date type: <class 'datetime.datetime'>
trade_date tzinfo: UTC
volume: 42243600.0
volume type: <class 'float'>


In [16]:
COLLECTION_NAME = "StockDailyPrice"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

prices = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="symbol",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="company_name",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="sector",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="trade_date",
            data_type=wvc.config.DataType.DATE,
        ),
        wvc.config.Property(
            name="open_price",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="high_price",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="low_price",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="close_price",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="volume",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="daily_change_pct",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="intraday_range_pct",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="trend_label",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="volatility_label",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="summary",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: StockDailyPrice


In [17]:
result = prices.data.insert_many(stock_records)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [18]:
response = prices.aggregate.over_all(total_count=True)

print("Total objects:", response.total_count)

Total objects: 50


In [19]:
response = prices.query.fetch_objects(
    limit=10,
    return_properties=[
        "symbol",
        "company_name",
        "trade_date",
        "open_price",
        "close_price",
        "volume",
        "daily_change_pct",
        "trend_label",
        "volatility_label",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Symbol:", obj.properties["symbol"])
    print("Company:", obj.properties["company_name"])
    print("Date:", obj.properties["trade_date"])
    print("Open:", obj.properties["open_price"])
    print("Close:", obj.properties["close_price"])
    print("Volume:", obj.properties["volume"])
    print("Change %:", obj.properties["daily_change_pct"])
    print("Trend:", obj.properties["trend_label"])
    print("Volatility:", obj.properties["volatility_label"])
    print("-" * 80)

UUID: 0babfdbe-1986-49b3-b90a-7df97dcbd478
Symbol: TSLA
Company: Tesla Inc.
Date: 2026-05-21 13:30:00+00:00
Open: 422.17999267578125
Close: 417.8500061035156
Volume: 42636900.0
Change %: 0.1414
Trend: flat or small move
Volatility: high intraday range
--------------------------------------------------------------------------------
UUID: 1ad6b269-9015-4228-af1b-0998eedd28fb
Symbol: MSFT
Company: Microsoft Corporation
Date: 2026-05-26 13:30:00+00:00
Open: 416.42999267578125
Close: 416.0299987792969
Volume: 30398000.0
Change %: -0.6068
Trend: moderate loss
Volatility: normal intraday range
--------------------------------------------------------------------------------
UUID: 30c3d6b7-7615-4b2f-8417-d0a4bfdb3a48
Symbol: AAPL
Company: Apple Inc.
Date: 2026-05-29 13:30:00+00:00
Open: 311.7799987792969
Close: 312.05999755859375
Volume: 70026800.0
Change %: -0.144
Trend: flat or small move
Volatility: normal intraday range
-----------------------------------------------------------------------

In [20]:
response = prices.query.near_text(
    query="semiconductor stock with strong gain and high trading activity",
    limit=8,
    return_properties=[
        "symbol",
        "company_name",
        "sector",
        "trade_date",
        "close_price",
        "volume",
        "daily_change_pct",
        "intraday_range_pct",
        "trend_label",
        "volatility_label",
        "summary",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Symbol:", obj.properties["symbol"])
    print("Company:", obj.properties["company_name"])
    print("Sector:", obj.properties["sector"])
    print("Date:", obj.properties["trade_date"])
    print("Change %:", obj.properties["daily_change_pct"])
    print("Range %:", obj.properties["intraday_range_pct"])
    print("Trend:", obj.properties["trend_label"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Distance: 0.5068603157997131
Symbol: NVDA
Company: NVIDIA Corporation
Sector: Semiconductors
Date: 2026-06-01 13:30:00+00:00
Change %: 6.2612
Range %: 4.2513
Trend: strong gain
Summary: NVIDIA Corporation (NVDA) traded on 2026-06-01 with open 215.73, high 224.87, low 215.70, close 224.36, volume 212850700. The daily change was 6.26% and the intraday range was 4.25%. This session is classified as strong gain with high intraday range.
--------------------------------------------------------------------------------
Distance: 0.543990969657898
Symbol: NVDA
Company: NVIDIA Corporation
Sector: Semiconductors
Date: 2026-05-28 13:30:00+00:00
Change %: 0.7761
Range %: 2.0358
Trend: moderate gain
Summary: NVIDIA Corporation (NVDA) traded on 2026-05-28 with open 211.28, high 215.52, low 211.22, close 214.25, volume 143996000. The daily change was 0.78% and the intraday range was 2.04%. This session is classified as moderate gain with normal intraday range.
----------------------------------------

In [21]:
response = prices.query.fetch_objects(
    filters=Filter.by_property("symbol").equal("NVDA"),
    limit=20,
    return_properties=[
        "symbol",
        "company_name",
        "trade_date",
        "close_price",
        "volume",
        "daily_change_pct",
        "trend_label",
    ],
)

for obj in response.objects:
    print("Symbol:", obj.properties["symbol"])
    print("Date:", obj.properties["trade_date"])
    print("Close:", obj.properties["close_price"])
    print("Volume:", obj.properties["volume"])
    print("Change %:", obj.properties["daily_change_pct"])
    print("Trend:", obj.properties["trend_label"])
    print("-" * 80)

Symbol: NVDA
Date: 2026-05-19 13:30:00+00:00
Close: 220.61000061035156
Volume: 140948200.0
Change %: 0.4508
Trend: flat or small move
--------------------------------------------------------------------------------
Symbol: NVDA
Date: 2026-05-20 13:30:00+00:00
Close: 223.47000122070312
Volume: 184201600.0
Change %: 1.2964
Trend: moderate gain
--------------------------------------------------------------------------------
Symbol: NVDA
Date: 2026-05-21 13:30:00+00:00
Close: 219.50999450683594
Volume: 203381800.0
Change %: -1.7721
Trend: moderate loss
--------------------------------------------------------------------------------
Symbol: NVDA
Date: 2026-05-22 13:30:00+00:00
Close: 215.3300018310547
Volume: 169275700.0
Change %: -1.9042
Trend: moderate loss
--------------------------------------------------------------------------------
Symbol: NVDA
Date: 2026-05-26 13:30:00+00:00
Close: 214.86000061035156
Volume: 187202600.0
Change %: -0.2183
Trend: flat or small move
-------------------

In [22]:
response = prices.query.fetch_objects(
    filters=Filter.by_property("daily_change_pct").greater_or_equal(2.0),
    limit=20,
    return_properties=[
        "symbol",
        "company_name",
        "trade_date",
        "close_price",
        "daily_change_pct",
        "trend_label",
        "summary",
    ],
)

for obj in response.objects:
    print("Symbol:", obj.properties["symbol"])
    print("Company:", obj.properties["company_name"])
    print("Date:", obj.properties["trade_date"])
    print("Close:", obj.properties["close_price"])
    print("Change %:", obj.properties["daily_change_pct"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Symbol: AAPL
Company: Apple Inc.
Date: 2026-06-02 13:30:00+00:00
Close: 315.20001220703125
Change %: 2.9023
Summary: Apple Inc. (AAPL) traded on 2026-06-02 with open 307.46, high 315.45, low 306.69, close 315.20, volume 44416900. The daily change was 2.90% and the intraday range was 2.86%. This session is classified as strong gain with normal intraday range.
--------------------------------------------------------------------------------
Symbol: MSFT
Company: Microsoft Corporation
Date: 2026-05-28 13:30:00+00:00
Close: 426.989990234375
Change %: 3.4701
Summary: Microsoft Corporation (MSFT) traded on 2026-05-28 with open 412.98, high 429.49, low 412.67, close 426.99, volume 47250500. The daily change was 3.47% and the intraday range was 4.08%. This session is classified as strong gain with high intraday range.
--------------------------------------------------------------------------------
Symbol: MSFT
Company: Microsoft Corporation
Date: 2026-05-29 13:30:00+00:00
Close: 450.23999023437

In [23]:
response = prices.query.fetch_objects(
    filters=Filter.by_property("intraday_range_pct").greater_or_equal(3.0),
    limit=20,
    return_properties=[
        "symbol",
        "company_name",
        "trade_date",
        "intraday_range_pct",
        "volatility_label",
        "summary",
    ],
)

for obj in response.objects:
    print("Symbol:", obj.properties["symbol"])
    print("Company:", obj.properties["company_name"])
    print("Date:", obj.properties["trade_date"])
    print("Intraday range %:", obj.properties["intraday_range_pct"])
    print("Volatility:", obj.properties["volatility_label"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Symbol: MSFT
Company: Microsoft Corporation
Date: 2026-05-19 13:30:00+00:00
Intraday range %: 3.8921
Volatility: high intraday range
Summary: Microsoft Corporation (MSFT) traded on 2026-05-19 with open 429.90, high 432.70, low 416.49, close 417.42, volume 33018700. The daily change was -2.90% and the intraday range was 3.89%. This session is classified as strong loss with high intraday range.
--------------------------------------------------------------------------------
Symbol: MSFT
Company: Microsoft Corporation
Date: 2026-05-28 13:30:00+00:00
Intraday range %: 4.0759
Volatility: high intraday range
Summary: Microsoft Corporation (MSFT) traded on 2026-05-28 with open 412.98, high 429.49, low 412.67, close 426.99, volume 47250500. The daily change was 3.47% and the intraday range was 4.08%. This session is classified as strong gain with high intraday range.
--------------------------------------------------------------------------------
Symbol: MSFT
Company: Microsoft Corporation
Dat

In [24]:
response = prices.query.hybrid(
    query="AI semiconductor stock strong gain NVIDIA",
    alpha=0.5,
    limit=8,
    return_properties=[
        "symbol",
        "company_name",
        "sector",
        "trade_date",
        "close_price",
        "daily_change_pct",
        "intraday_range_pct",
        "summary",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Symbol:", obj.properties["symbol"])
    print("Company:", obj.properties["company_name"])
    print("Sector:", obj.properties["sector"])
    print("Date:", obj.properties["trade_date"])
    print("Change %:", obj.properties["daily_change_pct"])
    print("Range %:", obj.properties["intraday_range_pct"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Score: 1.0
Symbol: NVDA
Company: NVIDIA Corporation
Sector: Semiconductors
Date: 2026-06-01 13:30:00+00:00
Change %: 6.2612
Range %: 4.2513
Summary: NVIDIA Corporation (NVDA) traded on 2026-06-01 with open 215.73, high 224.87, low 215.70, close 224.36, volume 212850700. The daily change was 6.26% and the intraday range was 4.25%. This session is classified as strong gain with high intraday range.
--------------------------------------------------------------------------------
Score: 0.712004542350769
Symbol: NVDA
Company: NVIDIA Corporation
Sector: Semiconductors
Date: 2026-05-28 13:30:00+00:00
Change %: 0.7761
Range %: 2.0358
Summary: NVIDIA Corporation (NVDA) traded on 2026-05-28 with open 211.28, high 215.52, low 211.22, close 214.25, volume 143996000. The daily change was 0.78% and the intraday range was 2.04%. This session is classified as moderate gain with normal intraday range.
--------------------------------------------------------------------------------
Score: 0.70134460926

In [25]:
response = prices.generate.near_text(
    query="stocks with strong gains and notable market movement",
    filters=Filter.by_property("daily_change_pct").greater_or_equal(1.0),
    grouped_task=(
        "Create a concise educational market-style summary using only the retrieved stock price records. "
        "Mention symbols, company names, trade dates, daily percentage changes, and whether the move looked strong or moderate. "
        "Do not give financial advice. Add a short note that this is not investment advice."
    ),
    limit=10,
    return_properties=[
        "symbol",
        "company_name",
        "sector",
        "trade_date",
        "close_price",
        "daily_change_pct",
        "intraday_range_pct",
        "trend_label",
        "summary",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["symbol"],
        "|",
        obj.properties["trade_date"],
        "| change:",
        obj.properties["daily_change_pct"],
        "%",
    )

### Market Summary

#### Microsoft Corporation (MSFT)
- **Trade Date:** 2026-06-01
  - **Open:** 464.84 | **High:** 466.32 | **Low:** 458.27 | **Close:** 460.52
  - **Daily Change:** +2.28% (Strong Gain)
  - **Intraday Range:** 1.76% (Normal)

- **Trade Date:** 2026-05-29
  - **Open:** 432.55 | **High:** 450.33 | **Low:** 432.36 | **Close:** 450.24
  - **Daily Change:** +5.45% (Strong Gain)
  - **Intraday Range:** 4.16% (High)

- **Trade Date:** 2026-05-28
  - **Open:** 412.98 | **High:** 429.49 | **Low:** 412.67 | **Close:** 426.99
  - **Daily Change:** +3.47% (Strong Gain)
  - **Intraday Range:** 4.08% (High)

#### Amazon.com Inc. (AMZN)
- **Trade Date:** 2026-05-20
  - **Open:** 260.05 | **High:** 265.58 | **Low:** 259.53 | **Close:** 265.01
  - **Daily Change:** +2.19% (Strong Gain)
  - **Intraday Range:** 2.33% (Normal)

- **Trade Date:** 2026-05-27
  - **Open:** 266.15 | **High:** 272.41 | **Low:** 265.70 | **Close:** 271.85
  - **Daily Change:** +2.47% (Strong Gain)
  - **Intrad

In [26]:
def analyze_stock(
    symbol: str,
    *,
    limit: int = 10,
) -> None:
    response = prices.generate.near_text(
        query=f"recent trading behavior for {symbol}",
        filters=Filter.by_property("symbol").equal(symbol),
        grouped_task=(
            "Analyze the retrieved daily stock price records in an educational way. "
            "Use only the retrieved records. "
            "Mention the dates, closing prices, daily percentage changes, trend labels and volatility labels. "
            "Do not recommend buying or selling. State clearly that this is not investment advice."
        ),
        limit=limit,
        return_properties=[
            "symbol",
            "company_name",
            "sector",
            "trade_date",
            "open_price",
            "high_price",
            "low_price",
            "close_price",
            "volume",
            "daily_change_pct",
            "intraday_range_pct",
            "trend_label",
            "volatility_label",
            "summary",
        ],
    )

    print("SYMBOL:", symbol)
    print("-" * 80)

    print(response.generated)

    print("\nRecords used:")
    for obj in response.objects:
        print(
            "-",
            obj.properties["symbol"],
            "|",
            obj.properties["trade_date"],
            "| close:",
            obj.properties["close_price"],
            "| change:",
            obj.properties["daily_change_pct"],
            "%",
        )

In [27]:
analyze_stock("NVDA")

SYMBOL: NVDA
--------------------------------------------------------------------------------
Here is an analysis of the retrieved daily stock price records for NVIDIA Corporation (NVDA) over a series of trading days. The information provided below includes the trade dates, closing prices, daily percentage changes, trend labels, and volatility labels.

### Stock Price Records for NVIDIA Corporation (NVDA)

1. **Date:** 2026-06-02
   - **Closing Price:** $222.82
   - **Daily Percentage Change:** -0.6864%
   - **Trend Label:** Moderate Loss
   - **Volatility Label:** High Intraday Range

2. **Date:** 2026-06-01
   - **Closing Price:** $224.36
   - **Daily Percentage Change:** 6.2612%
   - **Trend Label:** Strong Gain
   - **Volatility Label:** High Intraday Range

3. **Date:** 2026-05-29
   - **Closing Price:** $211.14
   - **Daily Percentage Change:** -1.4516%
   - **Trend Label:** Moderate Loss
   - **Volatility Label:** High Intraday Range

4. **Date:** 2026-05-28
   - **Closing Price

In [28]:
analyze_stock("AAPL")

SYMBOL: AAPL
--------------------------------------------------------------------------------
Here’s an analysis of the retrieved daily stock price records for Apple Inc. (AAPL), highlighting the closing prices, daily percentage changes, trend labels, and volatility labels. It’s important to remember that this analysis is for educational purposes only and should not be considered as investment advice.

### Daily Stock Price Records for Apple Inc. (AAPL)

1. **Date:** 2026-05-19  
   **Closing Price:** $298.97  
   **Daily Percentage Change:** +0.6735%  
   **Trend Label:** Moderate Gain  
   **Volatility Label:** Normal Intraday Range  

2. **Date:** 2026-05-20  
   **Closing Price:** $302.25  
   **Daily Percentage Change:** +1.0971%  
   **Trend Label:** Moderate Gain  
   **Volatility Label:** Normal Intraday Range  

3. **Date:** 2026-05-21  
   **Closing Price:** $304.99  
   **Daily Percentage Change:** +0.9065%  
   **Trend Label:** Moderate Gain  
   **Volatility Label:** Norma

In [29]:
analyze_stock("TSLA")

SYMBOL: TSLA
--------------------------------------------------------------------------------
Let's analyze the daily stock price records for Tesla Inc. (TSLA) for the given dates. This analysis includes the closing prices, daily percentage changes, trend labels, and volatility labels.

### Daily Stock Price Records Analysis for Tesla Inc. (TSLA)

1. **Date:** 2026-05-20
   - **Closing Price:** $417.26
   - **Daily Change (%):** +3.25%
   - **Trend Label:** Strong Gain
   - **Volatility Label:** Normal Intraday Range

2. **Date:** 2026-05-21
   - **Closing Price:** $417.85
   - **Daily Change (%):** +0.14%
   - **Trend Label:** Flat or Small Move
   - **Volatility Label:** High Intraday Range

3. **Date:** 2026-05-22
   - **Closing Price:** $426.01
   - **Daily Change (%):** +1.95%
   - **Trend Label:** Moderate Gain
   - **Volatility Label:** Normal Intraday Range

4. **Date:** 2026-05-26
   - **Closing Price:** $433.59
   - **Daily Change (%):** +1.78%
   - **Trend Label:** Moderate 

In [30]:
def find_market_sessions(
    query: str,
    *,
    sector: str | None = None,
    min_change_pct: float | None = None,
    min_intraday_range_pct: float | None = None,
    limit: int = 10,
) -> None:
    filters = None

    if sector is not None:
        filters = Filter.by_property("sector").equal(sector)

    if min_change_pct is not None:
        change_filter = Filter.by_property("daily_change_pct").greater_or_equal(
            min_change_pct
        )
        filters = change_filter if filters is None else filters & change_filter

    if min_intraday_range_pct is not None:
        range_filter = Filter.by_property("intraday_range_pct").greater_or_equal(
            min_intraday_range_pct
        )
        filters = range_filter if filters is None else filters & range_filter

    response = prices.query.near_text(
        query=query,
        filters=filters,
        limit=limit,
        return_properties=[
            "symbol",
            "company_name",
            "sector",
            "trade_date",
            "close_price",
            "volume",
            "daily_change_pct",
            "intraday_range_pct",
            "trend_label",
            "volatility_label",
            "summary",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    print("QUERY:", query)
    print("SECTOR:", sector)
    print("MIN CHANGE:", min_change_pct)
    print("MIN RANGE:", min_intraday_range_pct)
    print("-" * 80)

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Symbol:", obj.properties["symbol"])
        print("Company:", obj.properties["company_name"])
        print("Sector:", obj.properties["sector"])
        print("Date:", obj.properties["trade_date"])
        print("Close:", obj.properties["close_price"])
        print("Change %:", obj.properties["daily_change_pct"])
        print("Range %:", obj.properties["intraday_range_pct"])
        print("Trend:", obj.properties["trend_label"])
        print("Volatility:", obj.properties["volatility_label"])
        print("-" * 80)

In [31]:
find_market_sessions(
    "technology stock with positive market movement",
    sector="Technology",
    min_change_pct=0.5,
)

QUERY: technology stock with positive market movement
SECTOR: Technology
MIN CHANGE: 0.5
MIN RANGE: None
--------------------------------------------------------------------------------
Distance: 0.49291133880615234
Symbol: MSFT
Company: Microsoft Corporation
Sector: Technology
Date: 2026-05-29 13:30:00+00:00
Close: 450.239990234375
Change %: 5.4451
Range %: 4.1563
Trend: strong gain
Volatility: high intraday range
--------------------------------------------------------------------------------
Distance: 0.49383825063705444
Symbol: MSFT
Company: Microsoft Corporation
Sector: Technology
Date: 2026-05-28 13:30:00+00:00
Close: 426.989990234375
Change %: 3.4701
Range %: 4.0759
Trend: strong gain
Volatility: high intraday range
--------------------------------------------------------------------------------
Distance: 0.4972020387649536
Symbol: MSFT
Company: Microsoft Corporation
Sector: Technology
Date: 2026-06-01 13:30:00+00:00
Close: 460.5199890136719
Change %: 2.2832
Range %: 1.7566
Tren

In [32]:
find_market_sessions(
    "semiconductor stock with strong gain and volatility",
    sector="Semiconductors",
    min_change_pct=1.0,
    min_intraday_range_pct=2.0,
)

QUERY: semiconductor stock with strong gain and volatility
SECTOR: Semiconductors
MIN CHANGE: 1.0
MIN RANGE: 2.0
--------------------------------------------------------------------------------
Distance: 0.517028272151947
Symbol: NVDA
Company: NVIDIA Corporation
Sector: Semiconductors
Date: 2026-06-01 13:30:00+00:00
Close: 224.36000061035156
Change %: 6.2612
Range %: 4.2513
Trend: strong gain
Volatility: high intraday range
--------------------------------------------------------------------------------
Distance: 0.5609351396560669
Symbol: NVDA
Company: NVIDIA Corporation
Sector: Semiconductors
Date: 2026-05-20 13:30:00+00:00
Close: 223.47000122070312
Change %: 1.2964
Range %: 2.5533
Trend: moderate gain
Volatility: normal intraday range
--------------------------------------------------------------------------------


In [33]:
find_market_sessions(
    "automotive stock with volatile trading session",
    sector="Automotive",
    min_intraday_range_pct=2.0,
)

QUERY: automotive stock with volatile trading session
SECTOR: Automotive
MIN CHANGE: None
MIN RANGE: 2.0
--------------------------------------------------------------------------------
Distance: 0.48517030477523804
Symbol: TSLA
Company: Tesla Inc.
Sector: Automotive
Date: 2026-05-19 13:30:00+00:00
Close: 404.1099853515625
Change %: 0.2356
Range %: 3.0485
Trend: flat or small move
Volatility: high intraday range
--------------------------------------------------------------------------------
Distance: 0.4865958094596863
Symbol: TSLA
Company: Tesla Inc.
Sector: Automotive
Date: 2026-05-20 13:30:00+00:00
Close: 417.260009765625
Change %: 3.2541
Range %: 2.724
Trend: strong gain
Volatility: normal intraday range
--------------------------------------------------------------------------------
Distance: 0.4875909686088562
Symbol: TSLA
Company: Tesla Inc.
Sector: Automotive
Date: 2026-06-01 13:30:00+00:00
Close: 415.8800048828125
Change %: -4.5687
Range %: 3.4109
Trend: strong loss
Volatilit

In [34]:
client.close()